# LeetCode 621 — Task Scheduler bằng Python

Link đề: https://leetcode.com/problems/task-scheduler/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Hash Table, Counting, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def leastInterval(self, tasks: List[str], n: int) -> int:
        ...
```

Ghi chú: ví dụ 4, 5, 6 trong mục 2 do notebook tự đặt để minh hoạ các trường hợp biên, đáp án đã kiểm chứng bằng code (kể cả bằng tìm kiếm vét cạn); nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xếp một dãy công việc (mỗi việc là một chữ cái `A`–`Z`) vào các **khoảng thời gian CPU** liên tiếp; mỗi khoảng chạy đúng một việc hoặc **nghỉ (idle)**; hai việc **cùng nhãn** phải cách nhau **ít nhất `n` khoảng**; tìm **số khoảng tối thiểu** để xong tất cả.
2. Nắm các công cụ Python: `collections.Counter` để đếm tần suất, `max(...)` trên `values()`, `heapq` (min-heap, dùng số âm để thành max-heap), `collections.deque` làm hàng đợi "làm nguội", và phép tính chỉ số theo khung `n + 1`.
3. Đi từ **tìm kiếm vét cạn (BFS trên trạng thái)** → **mô phỏng tham lam với heap** → **mô phỏng theo từng "vòng" `n + 1` ô** → **công thức đóng `max(len, (x-1)(n+1)+s)`** → **cách đếm ô nghỉ cần lấp**.
4. Hiểu mấu chốt: **việc xuất hiện nhiều nhất** quyết định khung xương của lịch (`x - 1` khung dài `n + 1` ô, cộng phần đuôi); mọi việc còn lại chỉ để **lấp các ô trống**, và nếu quá nhiều việc thì **không có ô nghỉ nào** (đáp án = số việc).
5. Tránh các lỗi kinh điển: quên `max` với số việc, dùng `x * (n + 1)` thay vì `(x - 1) * (n + 1)`, quên cộng số việc **cùng đạt** tần suất lớn nhất, nhầm cooldown `n` với khoảng cách `n + 1`, và mô phỏng không cho nghỉ khi chưa việc nào sẵn sàng.

## 2. Đề bài & ví dụ

Cho mảng `tasks` gồm các việc của CPU, mỗi việc được gán nhãn bằng một chữ cái từ `A` đến `Z`, và một số nguyên `n`. Mỗi **khoảng CPU** (interval) hoặc rảnh rỗi (idle), hoặc hoàn thành **một** việc. Các việc có thể làm theo **thứ tự bất kỳ**, nhưng giữa hai việc **cùng nhãn** phải cách nhau **ít nhất `n` khoảng**.

Trả về **số khoảng CPU tối thiểu** để hoàn thành tất cả các việc.

**Ví dụ 1**

- Input: `tasks = ["A","A","A","B","B","B"]`, `n = 2`
- Output: `8`
- Giải thích: `A -> B -> idle -> A -> B -> idle -> A -> B`. Sau khi làm `A`, phải đợi 2 khoảng mới làm lại `A`; tương tự `B`.

**Ví dụ 2**

- Input: `tasks = ["A","C","A","B","D","B"]`, `n = 1`
- Output: `6`
- Giải thích: `A -> B -> C -> D -> A -> B`.

**Ví dụ 3**

- Input: `tasks = ["A","A","A","B","B","B"]`, `n = 3`
- Output: `10`
- Giải thích: `A -> B -> idle -> idle -> A -> B -> idle -> idle -> A -> B`.

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `tasks = ["A"]`, `n = 100`
- Output: `1` (chỉ một việc, không có gì phải đợi).

**Ví dụ 5 (tự đặt)**

- Input: `tasks = ["A","A"]`, `n = 2`
- Output: `4` (`A -> idle -> idle -> A`).

**Ví dụ 6 (tự đặt)**

- Input: `tasks = ["A","A","A","B"]`, `n = 0`
- Output: `4` (`n = 0` thì không cần đợi, đáp án là số việc).

**Ràng buộc:**

- `1 <= tasks.length <= 10^4`
- `tasks[i]` là một chữ cái in hoa tiếng Anh.
- `0 <= n <= 100`

In [1]:
from typing import List
from collections import Counter, deque
from heapq import heapify, heappop, heappush
import random
import time

examples = [
    (["A", "A", "A", "B", "B", "B"], 2, 8),
    (["A", "C", "A", "B", "D", "B"], 1, 6),
    (["A", "A", "A", "B", "B", "B"], 3, 10),
    (["A"], 100, 1),
    (["A", "A"], 2, 4),
    (["A", "A", "A", "B"], 0, 4),
]
for tasks, n, expected in examples:
    print(f"tasks={''.join(tasks)}, n={n} -> kỳ vọng {expected}")

tasks=AAABBB, n=2 -> kỳ vọng 8
tasks=ACABDB, n=1 -> kỳ vọng 6
tasks=AAABBB, n=3 -> kỳ vọng 10
tasks=A, n=100 -> kỳ vọng 1
tasks=AA, n=2 -> kỳ vọng 4
tasks=AAAB, n=0 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Đếm tần suất với `Counter`

`collections.Counter(iterable)` trả về một dict `{phần tử: số lần}`. `cnt.values()` là các số lần; `max(cnt.values())` là tần suất lớn nhất; `sum(1 for v in cnt.values() if v == x)` đếm có bao nhiêu nhãn đạt đúng tần suất `x`.

In [2]:
tasks = ["A", "A", "A", "B", "B", "C"]
cnt = Counter(tasks)
print(cnt)
x = max(cnt.values())
s = sum(1 for v in cnt.values() if v == x)
print("tần suất lớn nhất x =", x, "| số nhãn đạt x là s =", s)
print(cnt.most_common(2))

Counter({'A': 3, 'B': 2, 'C': 1})
tần suất lớn nhất x = 3 | số nhãn đạt x là s = 1
[('A', 3), ('B', 2)]


### 3.2. Heap (`heapq`) và max-heap bằng số âm

`heapq` trong Python là **min-heap**: `heappop` luôn lấy phần tử **nhỏ nhất**. Muốn lấy phần tử **lớn nhất**, ta đẩy **số âm** vào. `heapify(list)` biến list thành heap trong `O(n)`; `heappush`/`heappop` mỗi lần `O(log n)`.

In [3]:
counts = [3, 1, 2]
heap = [-c for c in counts]
heapify(heap)
print("lấy lớn nhất:", -heappop(heap))
heappush(heap, -5)
print("sau khi thêm 5, lớn nhất:", -heappop(heap))
print("còn lại:", sorted(-c for c in heap))

lấy lớn nhất: 3
sau khi thêm 5, lớn nhất: 5
còn lại: [1, 2]


### 3.3. `deque` làm hàng đợi "làm nguội"

`collections.deque` cho `append` (thêm cuối) và `popleft` (lấy đầu) đều `O(1)`. Ở cách mô phỏng, một việc vừa chạy sẽ **vào hàng đợi** kèm "thời điểm được chạy lại"; hàng đợi giữ đúng thứ tự thời gian nên chỉ cần nhìn phần tử đầu `queue[0]`.

In [4]:
q = deque()
q.append((3, "A"))
q.append((4, "B"))
print(q[0], len(q))
print(q.popleft(), q.popleft(), len(q))

(3, 'A') 2
(3, 'A') (4, 'B') 0


### 3.4. Khoảng cách giữa hai lần chạy và "khung" `n + 1`

Nếu một việc chạy ở khoảng thứ `t` thì lần chạy kế tiếp **sớm nhất** là khoảng `t + n + 1` (ở giữa có đúng `n` khoảng). Vì vậy một nhóm gồm **một việc và `n` khoảng sau nó** chiếm `n + 1` ô — ta gọi là một **khung (frame)** độ dài `n + 1`. Hai lần chạy của cùng một việc luôn nằm ở **hai khung khác nhau** hoặc cách nhau đủ xa.

In [5]:
n = 2
t = 0                                  # chạy A ở khoảng 0 (đếm từ 0)
print("lần chạy A tiếp theo sớm nhất ở khoảng", t + n + 1, "| ở giữa có", (t + n + 1) - t - 1, "khoảng")
# một khung: A _ _  (3 ô = n + 1)
print(("A" + "." * n) * 3)

lần chạy A tiếp theo sớm nhất ở khoảng 3 | ở giữa có 2 khoảng
A..A..A..


### 3.5. Phép `max` để "lấy cái nào lớn hơn trong hai cận"

Bài này có hai cận dưới cho đáp án: (1) ta không thể xong sớm hơn **số việc** (mỗi khoảng làm tối đa một việc); (2) ta không thể xong sớm hơn **khung xương** do việc xuất hiện nhiều nhất dựng nên. Đáp án là **cận nào lớn hơn**, tức `max(cận1, cận2)` — và ta sẽ chứng minh cả hai đều **đạt được**.

In [6]:
print(max(6, 7), max(len(["A"] * 9), 5))

7 9


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `tasks = [A, A, A, B, B, B]`, `n = 2`. Con người thường làm thế này: **vẽ các khung độ dài `n + 1 = 3`**, mỗi khung bắt đầu bằng các việc "nặng" nhất:

```
khung 1:  A B _
khung 2:  A B _
khung 3:  A B        (khung cuối không cần ô nghỉ)
```

Tổng: `3 + 3 + 2 = 8` khoảng. Ý tưởng: việc xuất hiện nhiều nhất (`A`, 3 lần) cần **ít nhất 3 khung** (mỗi khung một lần chạy), và các lần chạy `A` cách nhau đúng 3 ô. Việc `B` cũng 3 lần nên đi cùng `A` trong mỗi khung. Ô trống (`_`) ở khung 1 và 2 là ô nghỉ vì không còn việc nào để lấp.

Thử `n = 3` (`khung = 4`): `A B _ _ | A B _ _ | A B` → `4 + 4 + 2 = 10` ✓.

Thử `[A,A,A,B,B,B,C,C,D,E]`, `n = 2`: các khung `A B C | A B C | A B D`... còn `E` ta xếp vào ô trống. Số việc là 10, khung xương `(3-1)*3 + 2 = 8 < 10`, nên **không có ô nghỉ** và đáp án là `10`. Nhận xét: khi nhiều việc khác nhau, ta luôn lấp hết ô trống và đáp án chỉ bằng số việc.

In [7]:
def draw_frames(tasks, n):
    """Vẽ khung xương theo tần suất (chỉ để minh hoạ trực giác, không đảm bảo xếp đủ mọi việc)."""
    cnt = Counter(tasks)
    letters = [c for c, _ in cnt.most_common()]
    x = max(cnt.values())
    frames = []
    remaining = dict(cnt)
    for f in range(x):
        frame = []
        for c in letters:
            if remaining[c] > 0 and len(frame) < n + 1:
                frame.append(c)
                remaining[c] -= 1
        frames.append(frame)
    for k, fr in enumerate(frames, 1):
        pad = "_" * (n + 1 - len(fr)) if k < len(frames) else ""
        print(f"khung {k}: {' '.join(fr)} {' '.join(pad)}")

draw_frames(["A", "A", "A", "B", "B", "B"], 2)
print()
draw_frames(["A", "A", "A", "B", "B", "B"], 3)

khung 1: A B _
khung 2: A B _
khung 3: A B 

khung 1: A B _ _
khung 2: A B _ _
khung 3: A B 


## 5. Cách 1 — Brute Force / baseline: tìm kiếm vét cạn trên trạng thái (BFS)

### 5.1. Ý tưởng

Mô hình hoá thẳng đề: **trạng thái** tại một thời điểm gồm (a) số việc còn lại của mỗi nhãn `rem` và (b) với mỗi nhãn, số khoảng còn phải đợi trước khi được chạy lại `cool` (0 nghĩa là sẵn sàng). Mỗi khoảng ta có tối đa `k + 1` lựa chọn: chạy một nhãn sẵn sàng còn việc, hoặc nghỉ. Chạy nhãn `i`: `rem[i] -= 1`, `cool[i] = n`, các `cool` khác giảm 1 (không âm). **BFS** tìm số bước ít nhất để `rem` về toàn 0 — nghiệm **đúng tuyệt đối** vì thử mọi khả năng.

In [8]:
def least_interval_bfs(tasks, n):
    letters = sorted(set(tasks))
    rem0 = tuple(tasks.count(c) for c in letters)
    start = (rem0, (0,) * len(letters))
    queue = deque([(start, 0)])
    seen = {start}
    while queue:
        (rem, cool), t = queue.popleft()
        if not any(rem):
            return t
        dec = tuple(max(0, c - 1) for c in cool)
        nexts = [(rem, dec)]                                  # nghỉ
        for i in range(len(letters)):
            if rem[i] > 0 and cool[i] == 0:                   # chạy nhãn i
                r2, c2 = list(rem), list(dec)
                r2[i] -= 1
                c2[i] = n
                nexts.append((tuple(r2), tuple(c2)))
        for st in nexts:
            if st not in seen:
                seen.add(st)
                queue.append((st, t + 1))

for tasks, n, expected in examples:
    got = least_interval_bfs(tasks, n)
    print(f"{''.join(tasks)}, n={n} -> {got}")
    assert got == expected

AAABBB, n=2 -> 8
ACABDB, n=1 -> 6
AAABBB, n=3 -> 10
A, n=100 -> 1
AA, n=2 -> 4
AAAB, n=0 -> 4


### 5.2. Độ phức tạp và nút thắt

Số trạng thái tối đa cỡ `Π (rem_i + 1) · (n + 1)^k`, bùng nổ theo số nhãn `k` và `n`. Với `10^4` việc và 26 nhãn thì không thể. Hãy xem số trạng thái BFS đã duyệt tăng thế nào:

In [9]:
def bfs_states(tasks, n):
    letters = sorted(set(tasks))
    start = (tuple(tasks.count(c) for c in letters), (0,) * len(letters))
    queue, seen = deque([(start, 0)]), {start}
    while queue:
        (rem, cool), t = queue.popleft()
        if not any(rem):
            return len(seen)
        dec = tuple(max(0, c - 1) for c in cool)
        nexts = [(rem, dec)]
        for i in range(len(letters)):
            if rem[i] > 0 and cool[i] == 0:
                r2, c2 = list(rem), list(dec)
                r2[i] -= 1
                c2[i] = n
                nexts.append((tuple(r2), tuple(c2)))
        for st in nexts:
            if st not in seen:
                seen.add(st)
                queue.append((st, t + 1))

for tasks, n in [("AAB", 2), ("AABB", 2), ("AABBC", 2), ("AABBCC", 2), ("AABBCCD", 3), ("AABBCCDD", 3)]:
    t0 = time.perf_counter()
    states = bfs_states(list(tasks), n)
    print(f"{tasks:<9} n={n}: {states:7d} trạng thái, {1000 * (time.perf_counter() - t0):8.2f} ms")

AAB       n=2:      19 trạng thái,     0.06 ms
AABB      n=2:      33 trạng thái,     0.07 ms
AABBC     n=2:      99 trạng thái,     0.20 ms
AABBCC    n=2:     193 trạng thái,     0.37 ms
AABBCCD   n=3:    1023 trạng thái,     2.35 ms
AABBCCDD  n=3:    2301 trạng thái,     7.73 ms


Nút thắt: ta **không cần** nhớ chi tiết "việc nào đang nguội bao lâu" — cấu trúc bài rất đều, và chỉ cần biết **tần suất** là đủ. Các cách sau khai thác điều đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 — chỉ tần suất quan trọng.** Tên nhãn không ảnh hưởng, chỉ có **số lần** mỗi nhãn xuất hiện.

**Quan sát 2 — việc nhiều nhất là điểm nghẽn.** Gọi `x` là tần suất lớn nhất. Một nhãn xuất hiện `x` lần cần `x` lần chạy cách nhau `n + 1` ô → tối thiểu chiếm `(x - 1) · (n + 1) + 1` ô (từ lần đầu tới lần cuối). Nếu có `s` nhãn cùng tần suất `x`, mỗi nhãn đó cần thêm một ô ở cuối: tổng `(x - 1)(n + 1) + s`.

**Quan sát 3 — việc còn lại lấp ô trống.** Những việc khác (và các lần chạy "thừa") nhét vào các ô trống giữa các lần chạy việc nhiều nhất. Nếu **không đủ việc để lấp**, các ô trống thành ô nghỉ (idle). Nếu **dư việc**, thì không ô nào nghỉ và ta chỉ cần `len(tasks)` ô (vì số việc là cận dưới hiển nhiên).

Kết luận: `đáp án = max(len(tasks), (x - 1)(n + 1) + s)`.

Trước khi tin công thức, ta tìm hiểu cách làm **mô phỏng** (tham lam) cho thấy nó đúng ở mức thao tác, rồi chứng minh công thức ở mục 10.

## 7. Các cách tiếp theo

Tổng cộng có 5 cách: Cách 1 (BFS vét cạn, mục 5) và bốn cách dưới đây. Mỗi cách: giải thích từng đoạn → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Mô phỏng tham lam với max-heap + hàng đợi làm nguội

Ý tưởng: ở mỗi khoảng, **làm việc còn nhiều nhất** trong số các việc **sẵn sàng**; việc vừa làm (nếu còn) vào hàng đợi làm nguội đến khi được chạy lại. Nếu chưa việc nào sẵn sàng thì **nghỉ**. Tham lam theo "còn nhiều nhất" vì việc nhiều nhất là điểm nghẽn, nên phải làm sớm nhất có thể.

Giải thích từng đoạn:

1. `heap` chứa số âm của số việc còn lại (max-heap) của các nhãn **sẵn sàng**.
2. `cool` là `deque` các cặp `(t_ready, remaining)`; `t_ready` là khoảng mà đến cuối khoảng đó việc **được trả về heap**.
3. Mỗi khoảng `t`: nếu `heap` có việc, lấy việc có `remaining` lớn nhất, giảm 1; nếu vẫn còn thì đưa vào `cool` với `t_ready = t + n` (để khoảng `t + n + 1` nó đã sẵn sàng). Nếu `heap` rỗng, đó là khoảng nghỉ.
4. Cuối khoảng `t`, nếu `cool[0]` có `t_ready == t` thì trả về `heap`.
5. Lặp tới khi `heap` và `cool` đều rỗng; `t` chính là đáp án.

In [10]:
def schedule_heap(tasks, n):
    """Trả về danh sách lịch cụ thể (mỗi phần tử là nhãn hoặc 'idle') theo cách mô phỏng tham lam."""
    cnt = Counter(tasks)
    heap = [(-c, ch) for ch, c in cnt.items()]
    heapify(heap)
    cool = deque()
    seq = []
    t = 0
    while heap or cool:
        t += 1
        if heap:
            negc, ch = heappop(heap)
            seq.append(ch)
            if negc + 1 < 0:
                cool.append((t + n, negc + 1, ch))
        else:
            seq.append("idle")
        if cool and cool[0][0] == t:
            _, negc, ch = cool.popleft()
            heappush(heap, (negc, ch))
    return seq

for tasks, n, expected in examples:
    seq = schedule_heap(tasks, n)
    print(f"n={n}:", " ".join(seq))
    assert len(seq) == expected

n=2: A B idle A B idle A B
n=1: A B A B C D
n=3: A B idle idle A B idle idle A B
n=100: A
n=2: A idle idle A
n=0: A A A B


In [11]:
class Sol2:
    """Cách 2: mô phỏng tham lam với max-heap và hàng đợi làm nguội (nhảy qua khoảng nghỉ)."""
    def leastInterval(self, tasks: List[str], n: int) -> int:
        heap = [-c for c in Counter(tasks).values()]
        heapify(heap)
        cool = deque()                     # (t_ready, -số việc còn lại)
        t = 0
        while heap or cool:
            if not heap:                   # chưa việc nào sẵn sàng: nhảy tới lúc việc đầu hàng đợi sẵn sàng
                t = cool[0][0]
                heappush(heap, cool.popleft()[1])
                continue
            t += 1
            negc = heappop(heap) + 1
            if negc:
                cool.append((t + n, negc))
            if cool and cool[0][0] == t:
                heappush(heap, cool.popleft()[1])
        return t

for tasks, n, expected in examples:
    assert Sol2().leastInterval(tasks, n) == expected
print("Sol2 OK")

Sol2 OK


**Độ phức tạp:** mỗi lần chạy một việc tốn `O(log k)` (`k ≤ 26` nhãn) và có `T = len(tasks)` lần chạy; các khoảng nghỉ được **nhảy qua** (mỗi lần nhảy kèm một lần đẩy vào heap) nên không phụ thuộc `n`. Time `O(T log k)`, Space `O(k)`. (Bản `schedule_heap` ở trên đi từng khoảng nên thêm `O(số khoảng nghỉ)`.)

### 7.2. Cách 3 — Mô phỏng theo từng "vòng" `n + 1` ô

Một cách nhìn khác của tham lam: gom thời gian thành các **vòng** dài `n + 1` ô. Trong mỗi vòng, lấy tối đa `n + 1` **nhãn khác nhau** có số việc còn nhiều nhất, làm mỗi nhãn một lần (nên không nhãn nào lặp lại trong cùng một vòng, thoả cooldown); sau vòng, đẩy các nhãn còn việc về heap.

1. Lặp khi heap còn phần tử.
2. Trong vòng: lấy tối đa `n + 1` phần tử khác nhau (`taken` là số đã lấy), giảm 1 mỗi cái, cái nào còn thì cất vào `tmp`.
3. Đẩy `tmp` lại vào heap.
4. Cộng thời gian: nếu **vẫn còn** việc (heap không rỗng) thì vòng này chiếm đủ `n + 1` ô (kể cả ô nghỉ); nếu **đã hết** việc thì chỉ chiếm `taken` ô.

In [12]:
def rounds_demo(tasks, n):
    heap = [-c for c in Counter(tasks).values()]
    heapify(heap)
    time_used = 0
    r = 0
    while heap:
        r += 1
        tmp, taken = [], 0
        for _ in range(n + 1):
            if heap:
                c = heappop(heap) + 1
                taken += 1
                if c:
                    tmp.append(c)
        for c in tmp:
            heappush(heap, c)
        add = n + 1 if heap else taken
        time_used += add
        print(f"  vòng {r}: làm {taken} nhãn, còn lại {sorted(-c for c in heap)} -> +{add} (tổng {time_used})")
    return time_used

print(rounds_demo(["A", "A", "A", "B", "B", "B"], 2))
print(rounds_demo(["A", "A", "A", "A", "B", "B", "C"], 2))

  vòng 1: làm 2 nhãn, còn lại [2, 2] -> +3 (tổng 3)
  vòng 2: làm 2 nhãn, còn lại [1, 1] -> +3 (tổng 6)
  vòng 3: làm 2 nhãn, còn lại [] -> +2 (tổng 8)
8
  vòng 1: làm 3 nhãn, còn lại [1, 3] -> +3 (tổng 3)
  vòng 2: làm 2 nhãn, còn lại [2] -> +3 (tổng 6)
  vòng 3: làm 1 nhãn, còn lại [1] -> +3 (tổng 9)
  vòng 4: làm 1 nhãn, còn lại [] -> +1 (tổng 10)
10


In [13]:
class Sol3:
    """Cách 3: mô phỏng theo từng vòng n + 1 ô."""
    def leastInterval(self, tasks: List[str], n: int) -> int:
        heap = [-c for c in Counter(tasks).values()]
        heapify(heap)
        total = 0
        while heap:
            tmp, taken = [], 0
            for _ in range(n + 1):
                if heap:
                    c = heappop(heap) + 1
                    taken += 1
                    if c:
                        tmp.append(c)
            for c in tmp:
                heappush(heap, c)
            total += n + 1 if heap else taken
        return total

for tasks, n, expected in examples:
    assert Sol3().leastInterval(tasks, n) == expected
print("Sol3 OK")

Sol3 OK


**Độ phức tạp:** số vòng cỡ `x` (tần suất lớn nhất), mỗi vòng xử lý tối đa `min(n + 1, k)` phần tử `O(log k)`; tổng số lần lấy là `T`. Time `O(T log k)`, Space `O(k)`.

### 7.3. Cách 4 — Công thức đóng `max(len(tasks), (x - 1)(n + 1) + s)`

Từ quan sát ở mục 6: không cần mô phỏng, chỉ cần `x` và `s`.

1. `cnt = Counter(tasks)`.
2. `x = max(cnt.values())`; `s = số nhãn có tần suất đúng bằng x`.
3. Trả về `max(len(tasks), (x - 1) * (n + 1) + s)`.

Vì sao có `+ s`: sau `x - 1` khung đầy đủ, khung cuối chứa **một lần chạy của mỗi nhãn đạt tần suất `x`**, tức `s` ô.

In [14]:
def formula_demo(tasks, n):
    cnt = Counter(tasks)
    x = max(cnt.values())
    s = sum(1 for v in cnt.values() if v == x)
    skeleton = (x - 1) * (n + 1) + s
    print(f"  cnt={dict(cnt)} x={x} s={s} khung xương=({x}-1)*({n}+1)+{s}={skeleton} số việc={len(tasks)} -> {max(len(tasks), skeleton)}")
    return max(len(tasks), skeleton)

for tasks, n, expected in examples:
    assert formula_demo(tasks, n) == expected

  cnt={'A': 3, 'B': 3} x=3 s=2 khung xương=(3-1)*(2+1)+2=8 số việc=6 -> 8
  cnt={'A': 2, 'C': 1, 'B': 2, 'D': 1} x=2 s=2 khung xương=(2-1)*(1+1)+2=4 số việc=6 -> 6
  cnt={'A': 3, 'B': 3} x=3 s=2 khung xương=(3-1)*(3+1)+2=10 số việc=6 -> 10
  cnt={'A': 1} x=1 s=1 khung xương=(1-1)*(100+1)+1=1 số việc=1 -> 1
  cnt={'A': 2} x=2 s=1 khung xương=(2-1)*(2+1)+1=4 số việc=2 -> 4
  cnt={'A': 3, 'B': 1} x=3 s=1 khung xương=(3-1)*(0+1)+1=3 số việc=4 -> 4


In [15]:
class Sol4:
    """Cách 4: công thức đóng."""
    def leastInterval(self, tasks: List[str], n: int) -> int:
        cnt = Counter(tasks)
        x = max(cnt.values())
        s = sum(1 for v in cnt.values() if v == x)
        return max(len(tasks), (x - 1) * (n + 1) + s)

for tasks, n, expected in examples:
    assert Sol4().leastInterval(tasks, n) == expected
print("Sol4 OK")

Sol4 OK


**Độ phức tạp:** Time `O(T)` (đếm), Space `O(k)` với `k ≤ 26`, tức `O(1)`.

### 7.4. Cách 5 — Đếm ô nghỉ cần lấp (idle counting)

Cùng một đáp án nhưng suy luận theo hướng khác: **đáp án = số việc + số ô nghỉ**. Đặt khung xương theo việc nhiều nhất:

- Có `x - 1` khung đầy đủ dài `n + 1`; mỗi khung có `s` ô cho `s` nhãn tần suất `x`, còn **`n + 1 - s` ô trống** → tổng **ô trống ban đầu** `empty = (x - 1) · (n + 1 - s)`.
- Số việc **còn lại** để lấp các ô trống (không tính các lần chạy đã đặt vào khung xương, tức `x` lần cho mỗi trong `s` nhãn): `rest = len(tasks) - x · s`.
- Các ô trống mà **không lấp được** là ô nghỉ: `idle = max(0, empty - rest)`.
- Đáp án = `len(tasks) + idle`.

(Trường hợp `n + 1 - s < 0` — nhiều nhãn cùng tần suất lớn hơn cả khung — thì `empty` âm, `idle = 0`.) Đại số: `len + max(0, (x-1)(n+1-s) - (len - xs)) = max(len, (x-1)(n+1) + s)` — trùng công thức Cách 4, nhưng đây là cách hiểu "đếm lỗ trống" rất hay dùng khi giải thích.

In [16]:
def idle_demo(tasks, n):
    cnt = Counter(tasks)
    x = max(cnt.values())
    s = sum(1 for v in cnt.values() if v == x)
    empty = (x - 1) * (n + 1 - s)
    rest = len(tasks) - x * s
    idle = max(0, empty - rest)
    print(f"  x={x} s={s} | ô trống ban đầu={empty} | việc còn lại để lấp={rest} | idle={idle} | đáp án={len(tasks)}+{idle}={len(tasks) + idle}")
    return len(tasks) + idle

for tasks, n, expected in examples:
    assert idle_demo(tasks, n) == expected

  x=3 s=2 | ô trống ban đầu=2 | việc còn lại để lấp=0 | idle=2 | đáp án=6+2=8
  x=2 s=2 | ô trống ban đầu=0 | việc còn lại để lấp=2 | idle=0 | đáp án=6+0=6
  x=3 s=2 | ô trống ban đầu=4 | việc còn lại để lấp=0 | idle=4 | đáp án=6+4=10
  x=1 s=1 | ô trống ban đầu=0 | việc còn lại để lấp=0 | idle=0 | đáp án=1+0=1
  x=2 s=1 | ô trống ban đầu=2 | việc còn lại để lấp=0 | idle=2 | đáp án=2+2=4
  x=3 s=1 | ô trống ban đầu=0 | việc còn lại để lấp=1 | idle=0 | đáp án=4+0=4


In [17]:
class Sol5:
    """Cách 5: đếm ô nghỉ cần lấp."""
    def leastInterval(self, tasks: List[str], n: int) -> int:
        cnt = Counter(tasks)
        x = max(cnt.values())
        s = sum(1 for v in cnt.values() if v == x)
        empty = (x - 1) * (n + 1 - s)          # ô trống trong khung xương
        rest = len(tasks) - x * s              # việc còn lại có thể lấp vào ô trống
        return len(tasks) + max(0, empty - rest)

for tasks, n, expected in examples:
    assert Sol5().leastInterval(tasks, n) == expected
print("Sol5 OK")

Sol5 OK


**Độ phức tạp:** Time `O(T)`, Space `O(1)`.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 (heap + hàng đợi làm nguội)

Ví dụ `tasks = [A,A,A,B,B,B]`, `n = 2`. Bảng trạng thái (heap lưu số việc còn lại của nhãn sẵn sàng):

| `t` | việc chạy | heap sau khi chạy | hàng đợi `cool` (ready, còn lại) | heap sau khi trả về |
|---|---|---|---|---|
| 1 | A (3→2) | {B:3} | [(3, A:2)] | {B:3} |
| 2 | B (3→2) | {} | [(3, A:2), (4, B:2)] | {} |
| 3 | — (nghỉ) | {} | [(3, A:2), (4, B:2)] → trả A khi `t=3` | {A:2} |
| 4 | A (2→1) | {} | [(4, B:2), (6, A:1)] → trả B khi `t=4` | {B:2} |
| ... | ... | ... | ... | ... |

Bản đầy đủ do code in (lưu ý code ở dưới đi **từng khoảng**, nên in cả khoảng nghỉ):

In [18]:
def trace_heap(tasks, n):
    cnt = Counter(tasks)
    heap = [(-c, ch) for ch, c in cnt.items()]
    heapify(heap)
    cool = deque()
    t = 0
    seq = []
    print(f"tasks={''.join(tasks)}, n={n}")
    while heap or cool:
        t += 1
        if heap:
            negc, ch = heappop(heap)
            seq.append(ch)
            act = f"chạy {ch} ({-negc}->{-negc - 1})"
            if negc + 1 < 0:
                cool.append((t + n, negc + 1, ch))
                act += f", {ch} nguội tới cuối khoảng {t + n}"
        else:
            seq.append("idle")
            act = "nghỉ (heap rỗng)"
        back = ""
        if cool and cool[0][0] == t:
            _, negc, ch = cool.popleft()
            heappush(heap, (negc, ch))
            back = f" | trả {ch}({-negc}) về heap"
        print(f"t={t:>2}: {act:<42}{back}")
    print("lịch:", " ".join(seq), "| tổng", t)
    return t

trace_heap(["A", "A", "A", "B", "B", "B"], 2)

tasks=AAABBB, n=2
t= 1: chạy A (3->2), A nguội tới cuối khoảng 3  
t= 2: chạy B (3->2), B nguội tới cuối khoảng 4  
t= 3: nghỉ (heap rỗng)                           | trả A(2) về heap
t= 4: chạy A (2->1), A nguội tới cuối khoảng 6   | trả B(2) về heap
t= 5: chạy B (2->1), B nguội tới cuối khoảng 7  
t= 6: nghỉ (heap rỗng)                           | trả A(1) về heap
t= 7: chạy A (1->0)                              | trả B(1) về heap
t= 8: chạy B (1->0)                             
lịch: A B idle A B idle A B | tổng 8


8

Quan sát: ở `t = 3` heap rỗng → nghỉ; cuối `t = 3`, `A` (ready ở 3) trở lại heap nên `t = 4` làm `A`. Đúng tinh thần "lần chạy kế tiếp sớm nhất ở `t + n + 1`": `A` chạy ở `t = 1`, kế tiếp `t = 4`.

### 8.2. Trace Cách 3 (theo vòng) với nhiều nhãn

In [19]:
rounds_demo(["A", "A", "A", "B", "B", "B", "C", "C", "D"], 2)
print()
rounds_demo(["A", "A", "A", "A", "A", "B"], 2)

  vòng 1: làm 3 nhãn, còn lại [1, 1, 2, 2] -> +3 (tổng 3)
  vòng 2: làm 3 nhãn, còn lại [1, 1, 1] -> +3 (tổng 6)
  vòng 3: làm 3 nhãn, còn lại [] -> +3 (tổng 9)

  vòng 1: làm 2 nhãn, còn lại [4] -> +3 (tổng 3)
  vòng 2: làm 1 nhãn, còn lại [3] -> +3 (tổng 6)
  vòng 3: làm 1 nhãn, còn lại [2] -> +3 (tổng 9)
  vòng 4: làm 1 nhãn, còn lại [1] -> +3 (tổng 12)
  vòng 5: làm 1 nhãn, còn lại [] -> +1 (tổng 13)


13

Ở ví dụ thứ hai (`A` 5 lần, `B` 1 lần, `n = 2`): vòng đầu làm `A` và `B` rồi **nghỉ một ô**; các vòng sau chỉ có `A` nên mỗi vòng `A _ _` (2 ô nghỉ). Tổng `(5 - 1) · 3 + 1 = 13`.

### 8.3. Trace công thức (Cách 4) và đếm ô nghỉ (Cách 5)

Cùng một đầu vào, in cả hai cách để thấy chúng khớp:

In [20]:
for tasks, n in [(["A", "A", "A", "A", "A", "B"], 2), (["A", "A", "A", "B", "B", "B", "C", "C", "D"], 2), (["A", "B", "C", "D"], 5)]:
    print(f"tasks={''.join(tasks)}, n={n}")
    a = formula_demo(tasks, n)
    b = idle_demo(tasks, n)
    assert a == b
    print()

tasks=AAAAAB, n=2
  cnt={'A': 5, 'B': 1} x=5 s=1 khung xương=(5-1)*(2+1)+1=13 số việc=6 -> 13
  x=5 s=1 | ô trống ban đầu=8 | việc còn lại để lấp=1 | idle=7 | đáp án=6+7=13

tasks=AAABBBCCD, n=2
  cnt={'A': 3, 'B': 3, 'C': 2, 'D': 1} x=3 s=2 khung xương=(3-1)*(2+1)+2=8 số việc=9 -> 9
  x=3 s=2 | ô trống ban đầu=2 | việc còn lại để lấp=3 | idle=0 | đáp án=9+0=9

tasks=ABCD, n=5
  cnt={'A': 1, 'B': 1, 'C': 1, 'D': 1} x=1 s=4 khung xương=(1-1)*(5+1)+4=4 số việc=4 -> 4
  x=1 s=4 | ô trống ban đầu=0 | việc còn lại để lấp=0 | idle=0 | đáp án=4+0=4



### 8.4. Trace BFS trên một ví dụ nhỏ (nhìn lịch tối ưu)

Ta in một lịch tối ưu bằng cách truy ngược: lưu cha của mỗi trạng thái trong BFS.

In [21]:
def bfs_schedule(tasks, n):
    letters = sorted(set(tasks))
    start = (tuple(tasks.count(c) for c in letters), (0,) * len(letters))
    parent = {start: None}
    queue = deque([start])
    while queue:
        state = queue.popleft()
        rem, cool = state
        if not any(rem):
            seq = []
            while parent[state] is not None:
                state, move = parent[state]
                seq.append(move)
            return seq[::-1]
        dec = tuple(max(0, c - 1) for c in cool)
        moves = [("idle", (rem, dec))]
        for i, ch in enumerate(letters):
            if rem[i] > 0 and cool[i] == 0:
                r2, c2 = list(rem), list(dec)
                r2[i] -= 1
                c2[i] = n
                moves.append((ch, (tuple(r2), tuple(c2))))
        for move, st in moves:
            if st not in parent:
                parent[st] = (state, move)
                queue.append(st)

print(bfs_schedule(["A", "A", "A", "B", "B", "B"], 2))
print(bfs_schedule(["A", "A", "B"], 3))

['A', 'B', 'idle', 'A', 'B', 'idle', 'A', 'B']
['A', 'idle', 'idle', 'B', 'A']


## 9. Code nộp LeetCode

Dán đoạn dưới vào LeetCode (cho chắc, ta `import Counter` ở đầu; `List` đã có sẵn trên LeetCode).

In [22]:
from collections import Counter


class Solution:
    def leastInterval(self, tasks: List[str], n: int) -> int:
        cnt = Counter(tasks)
        x = max(cnt.values())                           # tần suất lớn nhất
        s = sum(1 for v in cnt.values() if v == x)      # số nhãn đạt tần suất đó
        return max(len(tasks), (x - 1) * (n + 1) + s)


for tasks, n, expected in examples:
    assert Solution().leastInterval(tasks, n) == expected
print("Solution chạy đúng 6 ví dụ")

Solution chạy đúng 6 ví dụ


## 10. Vì sao thuật toán đúng

Gọi `T = len(tasks)`, `x` là tần suất lớn nhất, `s` là số nhãn đạt `x`, và `F = (x - 1)(n + 1) + s`. Cần chứng minh đáp án tối ưu là `max(T, F)`.

### (A) Cận dưới: không thể ít hơn `max(T, F)`

- **`≥ T`:** mỗi khoảng làm tối đa một việc, cần `T` việc.
- **`≥ F`:** lấy một nhãn `L` có tần suất `x`. Các lần chạy `L` ở các khoảng `t_1 < t_2 < ... < t_x` thoả `t_{i+1} - t_i ≥ n + 1`, nên `t_x ≥ t_1 + (x - 1)(n + 1) ≥ 1 + (x - 1)(n + 1)`. Có `s` nhãn như vậy. Gọi `L'` là một nhãn khác trong `s` nhãn đó: lần chạy cuối của nó cũng `≥ 1 + (x-1)(n+1)`. Mỗi trong `s` nhãn đều có lần chạy cuối ở khoảng `≥ 1 + (x-1)(n+1)`, và các lần chạy cuối này ở các khoảng **khác nhau**, nên có ít nhất `s` khoảng phân biệt `≥ 1 + (x-1)(n+1)`; do đó khoảng lớn nhất `≥ (x-1)(n+1) + s = F`. Vậy tổng số khoảng `≥ F`.

### (B) Cận trên: có lịch đạt đúng `max(T, F)`

Nếu `x = 1` thì mọi nhãn khác nhau, xếp tuỳ ý, đáp án `T = F` (không có hai việc cùng nhãn). Giả sử `x ≥ 2`. Dựng `x - 1` **khung** và một **khung cuối**:

1. Mỗi khung (kể cả khung cuối) bắt đầu bằng `s` nhãn có tần suất `x` (cùng thứ tự). Mỗi nhãn này nằm ở cùng cột trong mọi khung.
2. Các nhãn còn lại (tần suất `c ≤ x - 1`) được thả **lần lượt từng bản sao** vào `x - 1` khung đầu theo vòng tròn: bản sao thứ 1 vào khung 1, thứ 2 vào khung 2, ..., khung `x - 1` rồi quay lại khung 1, và mỗi bản sao được **nối vào cuối** khung nhận nó (tức xếp theo "cột" trước, khung sau).
3. Khung nào còn ngắn hơn `n + 1` ô thì đệm thêm ô nghỉ ở cuối cho đủ `n + 1` ô. Nối tất cả các khung lại.

*Hợp lệ?* Các khung đầu đều có độ dài `≥ n + 1`. Xét hai bản sao của cùng một nhãn:

- Nhãn tần suất `x`: cùng cột ở hai khung liền nhau → cách nhau đúng độ dài một khung `≥ n + 1`. ✔
- Nhãn tần suất `c ≤ x - 1` (chỉ nằm trong `x - 1` khung đầu): hai bản sao ở hai **khung khác nhau** vì `c ≤ x - 1` và thả vòng tròn. Nếu cùng cột ở hai khung liền nhau thì cách nhau đúng độ dài một khung `≥ n + 1`. Nếu đi qua chỗ "quay vòng", một bản sao ở khung `f` cột `j + 1` và bản sao kia ở khung `f' > f` cột `j`; vì `c ≤ x - 1` nên `f' - f ≥ 2`, khoảng cách `≥ 2(n + 1) - 1 ≥ n + 1`. ✔

*Độ dài?* Nếu `F ≥ T` thì `s ≤ n + 1` (nếu không `T ≥ xs > F`), các khung đầu dài đúng `n + 1` (có thể có ô nghỉ), khung cuối dài `s` ⇒ tổng `(x - 1)(n + 1) + s = F`. Nếu `T > F` thì số việc nhiều hơn số ô của các khung dài `n + 1`, các khung đầu **dài hơn** `n + 1` và hoàn toàn không có ô nghỉ nào ⇒ tổng `T`. Vậy luôn đạt `max(T, F)`. ∎

Ta cũng cài đặt đúng cách dựng này ở mục 15.1 và kiểm tra thực nghiệm lịch dựng ra luôn hợp lệ.

### (C) Các cách khác

- **Cách 2, 3 (mô phỏng):** tham lam "làm việc còn nhiều nhất trong những việc sẵn sàng" là cách dựng đúng lịch ở (B); ta kiểm chứng bằng oracle BFS ở mục 12 trên mọi đầu vào nhỏ.
- **Cách 5:** đại số chứng tỏ bằng Cách 4 (đã nêu ở 7.4).
- **Cách 1 (BFS):** thử **mọi** lịch nên đúng theo định nghĩa; ta dùng nó làm **oracle**.

## 11. Phân tích Time / Space Complexity

Ký hiệu: `T = len(tasks)`, `k ≤ 26` số nhãn khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. BFS vét cạn | cực lớn (bùng nổ trạng thái) | cực lớn |
| 2. Heap + hàng đợi làm nguội | `O(T log k)` | `O(k)` |
| 3. Mô phỏng theo vòng `n + 1` | `O(T log k)` | `O(k)` |
| 4. Công thức đóng | `O(T)` | `O(k)` |
| 5. Đếm ô nghỉ | `O(T)` | `O(k)` |

`k ≤ 26` nên `log k` coi như hằng số. Đo thực nghiệm trên đầu vào lớn nhất (`T = 10^4`, `n = 100`):

In [23]:
rng = random.Random(621)
big = [rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(10**4)]
worst = ["A"] * 10**4                 # một nhãn duy nhất, n = 100: đáp án (10^4 - 1) * 101 + 1
for name, cls in [("Cách 2", Sol2), ("Cách 3", Sol3), ("Cách 4", Sol4), ("Cách 5", Sol5), ("Solution", Solution)]:
    t0 = time.perf_counter()
    r1 = cls().leastInterval(big, 100)
    t1 = time.perf_counter() - t0
    t0 = time.perf_counter()
    r2 = cls().leastInterval(worst, 100)
    t2 = time.perf_counter() - t0
    print(f"{name:<9}: ngẫu nhiên {1000 * t1:7.2f} ms (= {r1}) | một nhãn {1000 * t2:7.2f} ms (= {r2})")
assert r2 == (10**4 - 1) * 101 + 1

Cách 2   : ngẫu nhiên    4.15 ms (= 43128) | một nhãn    4.33 ms (= 1009900)


Cách 3   : ngẫu nhiên    3.32 ms (= 43128) | một nhãn   22.52 ms (= 1009900)
Cách 4   : ngẫu nhiên    0.50 ms (= 43128) | một nhãn    0.55 ms (= 1009900)
Cách 5   : ngẫu nhiên    0.49 ms (= 43128) | một nhãn    0.52 ms (= 1009900)
Solution : ngẫu nhiên    0.48 ms (= 43128) | một nhãn    0.58 ms (= 1009900)


## 12. Bộ test

### 12.1. Ví dụ đề và edge case

**Oracle độc lập:** BFS trạng thái (Cách 1) — không dùng công thức, heap hay tham lam. Chỉ chạy được trên đầu vào nhỏ nên các test vét cạn dùng đầu vào nhỏ. Ta gom các cách vào dict `solvers`.

In [24]:
solvers = {
    "bfs": least_interval_bfs,
    "heap": lambda t, n: Sol2().leastInterval(t, n),
    "rounds": lambda t, n: Sol3().leastInterval(t, n),
    "formula": lambda t, n: Sol4().leastInterval(t, n),
    "idle": lambda t, n: Sol5().leastInterval(t, n),
    "leetcode": lambda t, n: Solution().leastInterval(t, n),
}
for tasks, n, expected in examples:
    for name, fn in solvers.items():
        assert fn(tasks, n) == expected, (name, tasks, n)
print("Ví dụ đề: OK cho cả 6 solver (gồm cả BFS oracle)")

Ví dụ đề: OK cho cả 6 solver (gồm cả BFS oracle)


In [25]:
edge_cases = [
    (["A"], 0, 1, "một việc, n = 0"),
    (["A"], 100, 1, "một việc, n lớn"),
    (["A", "A"], 0, 2, "n = 0"),
    (["A", "A"], 1, 3, "A _ A"),
    (["A", "A", "A"], 1, 5, "A _ A _ A"),
    (["A", "B"], 5, 2, "hai nhãn khác nhau, không cần đợi"),
    (["A", "A", "B", "B"], 1, 4, "A B A B"),
    (["A", "A", "B", "B"], 2, 5, "A B _ A B"),
    (["A", "A", "A", "B"], 2, 7, "A B _ A _ _ A"),
    (["A", "A", "A", "A", "B", "B", "C"], 2, 10, "x = 4, s = 1"),
    (["A", "B", "C", "D"], 3, 4, "toàn nhãn khác nhau"),
    (["A", "A", "B", "B", "C", "C"], 2, 6, "nhiều nhãn cùng tần suất, lấp kín khung"),
    (["A", "A", "B", "B", "C", "C"], 3, 7, "khung 4, s = 3: A B C _ A B C"),
    (["A", "A", "A", "B", "B", "B", "C", "C", "C"], 1, 9, "s = 3 > n + 1 - ... không có ô nghỉ"),
    (["A", "A", "A", "B", "B", "B", "C", "C", "C", "D"], 2, 10, "nhiều việc, không idle"),
]
for tasks, n, expected, note in edge_cases:
    assert least_interval_bfs(tasks, n) == expected, ("oracle", note, least_interval_bfs(tasks, n))
    for name, fn in solvers.items():
        assert fn(tasks, n) == expected, (name, note)
print(f"{len(edge_cases)} edge case: OK")

15 edge case: OK


### 12.2. Phản ví dụ cho các ý tưởng sai

- **Thiếu `max` với số việc:** sai khi có quá nhiều việc so với khung xương, ví dụ `[A,A,B,B,C,C,D,D]`, `n = 1`: khung xương `(2-1)*2 + 4 = 6` nhưng có tới 8 việc nên đáp án là 8.
- **Dùng `x * (n + 1)`** thay vì `(x - 1) * (n + 1) + s`: dư ô nghỉ ở khung cuối.
- **Quên `+ s`** (chỉ cộng 1): sai khi nhiều nhãn cùng tần suất lớn nhất.

In [26]:
def wrong_no_max(tasks, n):
    cnt = Counter(tasks)
    x = max(cnt.values())
    s = sum(1 for v in cnt.values() if v == x)
    return (x - 1) * (n + 1) + s

def wrong_x_times(tasks, n):
    cnt = Counter(tasks)
    x = max(cnt.values())
    return max(len(tasks), x * (n + 1))

def wrong_plus_one(tasks, n):
    cnt = Counter(tasks)
    x = max(cnt.values())
    return max(len(tasks), (x - 1) * (n + 1) + 1)

cases = [(list("AABBCCDD"), 1), (list("AAB"), 2), (list("AABB"), 2)]
for tasks, n in cases:
    print(f"{''.join(tasks)} n={n} | đúng={least_interval_bfs(tasks, n)} | thiếu max={wrong_no_max(tasks, n)} | x*(n+1)={wrong_x_times(tasks, n)} | +1 thay +s={wrong_plus_one(tasks, n)}")
assert wrong_no_max(*cases[0]) != least_interval_bfs(*cases[0])
assert wrong_x_times(*cases[1]) != least_interval_bfs(*cases[1])
assert wrong_plus_one(*cases[2]) != least_interval_bfs(*cases[2])

AABBCCDD n=1 | đúng=8 | thiếu max=6 | x*(n+1)=8 | +1 thay +s=8
AAB n=2 | đúng=4 | thiếu max=4 | x*(n+1)=6 | +1 thay +s=4
AABB n=2 | đúng=5 | thiếu max=5 | x*(n+1)=6 | +1 thay +s=4


### 12.3. Đối chiếu mọi cách bằng vét cạn toàn bộ input nhỏ

Duyệt **mọi** multiset gồm tối đa 3 nhãn (`A, B, C`) với số lần `0..4` mỗi nhãn (tổng ≥ 1), và `n` từ `0..4`, so sánh 6 solver (gồm BFS oracle).

In [27]:
from itertools import product

checked = 0
for ca, cb, cc in product(range(5), repeat=3):
    if ca + cb + cc == 0:
        continue
    tasks = ["A"] * ca + ["B"] * cb + ["C"] * cc
    for n in range(5):
        expected = least_interval_bfs(tasks, n)
        for name, fn in solvers.items():
            assert fn(tasks, n) == expected, (name, tasks, n)
        checked += 1
print("đã kiểm", checked, "cặp (tasks, n): tất cả khớp oracle BFS")

đã kiểm 620 cặp (tasks, n): tất cả khớp oracle BFS


### 12.4. Test ngẫu nhiên có seed (đúng ràng buộc)

Với oracle BFS, đầu vào phải nhỏ: tối đa 4 nhãn và 8 việc, `n ≤ 5`. Thứ tự các việc trong `tasks` được xáo ngẫu nhiên để chắc chắn đáp án không phụ thuộc thứ tự.

In [28]:
rng = random.Random(621)
for _ in range(1500):
    k = rng.randint(1, 4)
    letters = rng.sample("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k)
    tasks = [rng.choice(letters) for _ in range(rng.randint(1, 8))]
    rng.shuffle(tasks)
    n = rng.randint(0, 5)
    assert 1 <= len(tasks) <= 10**4 and 0 <= n <= 100
    expected = least_interval_bfs(tasks, n)
    for name, fn in solvers.items():
        assert fn(tasks, n) == expected, (name, tasks, n)
print("1500 test ngẫu nhiên nhỏ: OK")

1500 test ngẫu nhiên nhỏ: OK


### 12.5. Test lớn và kiểm tra tính chất (lịch cụ thể hợp lệ)

Với `T` lớn không dùng BFS. Ta (a) so các cách `heap`, `rounds`, `formula`, `idle`, `leetcode` với nhau, và (b) dùng `schedule_heap` dựng lịch cụ thể rồi **kiểm tra bằng hàm độc lập** `valid_schedule` (đủ việc, mọi cặp cùng nhãn cách ≥ `n` khoảng) và độ dài đúng bằng đáp án.

In [29]:
def valid_schedule(seq, tasks, n):
    real = [x for x in seq if x != "idle"]
    if sorted(real) != sorted(tasks):
        return False
    last = {}
    for t, x in enumerate(seq):
        if x == "idle":
            continue
        if x in last and t - last[x] - 1 < n:       # số khoảng ở giữa phải >= n
            return False
        last[x] = t
    return True

rng = random.Random(6210)
fast = ["heap", "rounds", "formula", "idle", "leetcode"]
for trial in range(40):
    k = rng.choice([1, 2, 3, 10, 26])
    letters = rng.sample("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k)
    weights = [rng.random() ** rng.choice([1, 3]) for _ in letters]
    T = rng.choice([1, 50, 500, 3000])
    tasks = rng.choices(letters, weights=weights, k=T)
    n = rng.choice([0, 1, 2, 10, 50, 100])
    answers = {solvers[name](tasks, n) for name in fast}
    assert len(answers) == 1, (trial, answers)
    ans = answers.pop()
    seq = schedule_heap(tasks, n)
    assert valid_schedule(seq, tasks, n) and len(seq) == ans, trial
print("40 test lớn: 5 solver nhất quán, lịch cụ thể hợp lệ và có độ dài đúng bằng đáp án")

40 test lớn: 5 solver nhất quán, lịch cụ thể hợp lệ và có độ dài đúng bằng đáp án


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `max` với số việc

**Code sai:**

In [30]:
class WrongSolution1:
    def leastInterval(self, tasks, n):
        cnt = Counter(tasks)
        x = max(cnt.values())
        s = sum(1 for v in cnt.values() if v == x)
        return (x - 1) * (n + 1) + s          # thiếu max(len(tasks), ...)

for tasks, n in [(list("AABBCCDD"), 1), (list("ABCDEF"), 0), (list("AAABBBCCCDDD"), 2)]:
    print(f"{''.join(tasks)} n={n} | sai: {WrongSolution1().leastInterval(tasks, n)} | đúng: {Solution().leastInterval(tasks, n)}")

AABBCCDD n=1 | sai: 6 | đúng: 8
ABCDEF n=0 | sai: 6 | đúng: 6
AAABBBCCCDDD n=2 | sai: 10 | đúng: 12


**Vì sao sai:** khi có **quá nhiều việc** so với khung xương, các ô trống đều được lấp và còn dư việc nên tổng thời gian là **số việc**, lớn hơn `(x - 1)(n + 1) + s`. **Code đúng:** `max(len(tasks), (x - 1) * (n + 1) + s)`.

### Lỗi 2: Dùng `x * (n + 1)` (quên khung cuối ngắn hơn)

In [31]:
class WrongSolution2:
    def leastInterval(self, tasks, n):
        cnt = Counter(tasks)
        x = max(cnt.values())
        return max(len(tasks), x * (n + 1))      # sai: khung cuối không cần đủ n + 1 ô

for tasks, n in [(list("AAABBB"), 2), (list("AAB"), 2), (list("AAAA"), 1)]:
    print(f"{''.join(tasks)} n={n} | sai: {WrongSolution2().leastInterval(tasks, n)} | đúng: {Solution().leastInterval(tasks, n)}")

AAABBB n=2 | sai: 9 | đúng: 8
AAB n=2 | sai: 6 | đúng: 4
AAAA n=1 | sai: 8 | đúng: 7


**Vì sao sai:** sau lần chạy **cuối cùng** của việc nhiều nhất, ta **không cần đợi** thêm `n` ô nữa — công việc đã xong. Do đó `x - 1` khung đầy đủ cộng khung cuối chỉ gồm `s` việc. **Code đúng:** `(x - 1) * (n + 1) + s`.

### Lỗi 3: Chỉ cộng 1 thay vì `s` (bỏ qua các nhãn đồng hạng)

In [32]:
class WrongSolution3:
    def leastInterval(self, tasks, n):
        cnt = Counter(tasks)
        x = max(cnt.values())
        return max(len(tasks), (x - 1) * (n + 1) + 1)      # sai: nhiều nhãn đạt x thì phải +s

for tasks, n in [(list("AAABBB"), 2), (list("AABBCC"), 4), (list("AAAB"), 2)]:
    print(f"{''.join(tasks)} n={n} | sai: {WrongSolution3().leastInterval(tasks, n)} | đúng: {Solution().leastInterval(tasks, n)}")

AAABBB n=2 | sai: 7 | đúng: 8
AABBCC n=4 | sai: 6 | đúng: 8
AAAB n=2 | sai: 7 | đúng: 7


**Vì sao sai:** `[A,A,A,B,B,B]`, `n = 2`: `A` và `B` đều xuất hiện 3 lần nên khung cuối có **hai** việc (`A B`), tổng `2·3 + 2 = 8`, không phải `7`. **Code đúng:** `+ s` với `s` là số nhãn đạt tần suất lớn nhất.

### Lỗi 4: Nhầm khoảng cách `n` với `n + 1` (off-by-one)

In [33]:
class WrongSolution4:
    def leastInterval(self, tasks, n):
        cnt = Counter(tasks)
        x = max(cnt.values())
        s = sum(1 for v in cnt.values() if v == x)
        return max(len(tasks), (x - 1) * n + s)       # sai: khung dài n + 1, không phải n

for tasks, n in [(list("AAA"), 2), (list("AAABBB"), 2), (list("AA"), 1)]:
    print(f"{''.join(tasks)} n={n} | sai: {WrongSolution4().leastInterval(tasks, n)} | đúng: {Solution().leastInterval(tasks, n)}")

AAA n=2 | sai: 5 | đúng: 7
AAABBB n=2 | sai: 6 | đúng: 8
AA n=1 | sai: 2 | đúng: 3


**Vì sao sai:** "cách nhau ít nhất `n` khoảng" nghĩa là **giữa** hai lần chạy có `n` khoảng, nên **hiệu chỉ số** của hai lần chạy là `n + 1` — khung dài `n + 1` ô (một ô cho chính việc đó). `A _ _ A` có `n = 2` khoảng ở giữa và hai lần chạy cách nhau 3 ô. **Code đúng:** `(x - 1) * (n + 1)`.

### Lỗi 5: Mô phỏng không cho phép nghỉ (hoặc quên đưa việc về heap)

In [34]:
def wrong_sim_no_idle(tasks, n):
    heap = [-c for c in Counter(tasks).values()]
    heapify(heap)
    t = 0
    while heap:
        tmp = []
        for _ in range(n + 1):
            if heap:
                c = heappop(heap) + 1
                if c:
                    tmp.append(c)
            t += 1                         # sai: cộng cả khi heap đã hết, kể cả vòng cuối
        for c in tmp:
            heappush(heap, c)
    return t

for tasks, n in [(list("AAABBB"), 2), (list("AAAB"), 3), (list("AB"), 4)]:
    print(f"{''.join(tasks)} n={n} | sai: {wrong_sim_no_idle(tasks, n)} | đúng: {Solution().leastInterval(tasks, n)}")

AAABBB n=2 | sai: 9 | đúng: 8
AAAB n=3 | sai: 12 | đúng: 9
AB n=4 | sai: 5 | đúng: 2


**Vì sao sai:** vòng **cuối** không cần đủ `n + 1` ô: khi heap đã hết việc, ta chỉ tính số ô thực sự dùng (`taken`), không cộng ô nghỉ thừa. **Code đúng:** `total += n + 1 if heap else taken` (xem Cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"cùng loại phải cách nhau ít nhất k", "tối thiểu thời gian"** → nghĩ ngay **tần suất** (`Counter`) và **khung `n + 1`**; thường có công thức đóng.
- Bài **xếp lại chuỗi sao cho hai ký tự giống nhau không kề nhau / cách nhau `k`** (767, 358) cùng họ: cái gì nhiều nhất quyết định khả năng xếp được.
- Khi không nghĩ ra công thức, **mô phỏng tham lam với max-heap + hàng đợi làm nguội** gần như luôn dựng được lịch.
- Công thức hay gặp: `max(len, (x - 1) * (n + 1) + s)`. Hãy nhớ ba thành phần: `x - 1` khung đầy, `n + 1` độ dài khung, `s` ô ở khung cuối.
- "Cách nhau ít nhất `n`" nghĩa là **ở giữa có `n` khoảng** (hiệu chỉ số `n + 1`). Luôn kiểm tra off-by-one bằng ví dụ `A _ _ A`.
- Dùng **oracle BFS** trên input nhỏ để kiểm chứng công thức trước khi nộp — đó chính là cách ta làm ở mục 12.
- Luôn thử: một việc, `n = 0`, toàn nhãn khác nhau, nhiều nhãn cùng tần suất lớn nhất, một nhãn chiếm áp đảo.

## 15. Biến thể

### 15.1. Trả về lịch cụ thể (không chỉ độ dài)

Đây là thứ tự thực sự các việc/idle. Ta đã có `schedule_heap` ở mục 7.1 (kiểm chứng ở mục 12.5 bằng `valid_schedule`). Ở đây ta cài đặt thêm cách dựng **bằng khung** (không heap) đúng như chứng minh (B) ở mục 10, để thấy công thức đúng "bằng xây dựng".

In [35]:
def schedule_frames(tasks, n):
    cnt = Counter(tasks)
    x = max(cnt.values())
    if x == 1:
        return list(tasks)                           # mọi nhãn khác nhau
    maxl = sorted(c for c, v in cnt.items() if v == x)
    others = sorted((c for c, v in cnt.items() if v < x), key=lambda c: (-cnt[c], c))
    frames = [list(maxl) for _ in range(x - 1)]
    idx = 0
    for c in others:
        for _ in range(cnt[c]):
            frames[idx % (x - 1)].append(c)           # thả vòng tròn, nối vào cuối khung
            idx += 1
    seq = []
    for fr in frames:
        seq.extend(fr)
        seq.extend(["idle"] * (n + 1 - len(fr)))      # đệm ô nghỉ nếu khung ngắn hơn n + 1
    seq.extend(maxl)                                  # khung cuối
    return seq

for tasks, n in [(list("AAABBB"), 2), (list("AAAABBC"), 2), (list("AABBCCD"), 2)]:
    seq = schedule_frames(tasks, n)
    print(f"{''.join(tasks)} n={n}:", " ".join(seq), "| hợp lệ:", valid_schedule(seq, tasks, n), "| độ dài", len(seq), "| công thức", Solution().leastInterval(tasks, n))

rng = random.Random(151)
bad = 0
for _ in range(3000):
    k = rng.randint(1, 6)
    letters = rng.sample("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k)
    tasks = [rng.choice(letters) for _ in range(rng.randint(1, 30))]
    n = rng.randint(0, 8)
    seq = schedule_frames(tasks, n)
    ok = valid_schedule(seq, tasks, n) and len(seq) == Solution().leastInterval(tasks, n)
    bad += not ok
print("số ca lịch khung xương không hợp lệ hoặc sai độ dài:", bad)
assert bad == 0

AAABBB n=2: A B idle A B idle A B | hợp lệ: True | độ dài 8 | công thức 8
AAAABBC n=2: A B idle A B idle A C idle A | hợp lệ: True | độ dài 10 | công thức 10
AABBCCD n=2: A B C D A B C | hợp lệ: True | độ dài 7 | công thức 7
số ca lịch khung xương không hợp lệ hoặc sai độ dài: 0


Lịch dựng theo khung luôn hợp lệ và đạt đúng độ dài tối ưu — kiểm chứng bằng thực nghiệm cho phần (B) ở mục 10 (đặc biệt chú ý `idx % (x - 1)`: các nhãn tần suất nhỏ hơn `x` chỉ nằm trong `x - 1` khung đầu và không bao giờ có hai bản sao trong cùng một khung).

### 15.2. Số ô nghỉ tối thiểu và có thể xếp mà không cần nghỉ không?

Số ô nghỉ tối thiểu là `leastInterval - len(tasks)`. Bài **LeetCode 767 (Reorganize String)** là trường hợp `n = 1` với yêu cầu **không có ô nghỉ**: có thể xếp khi và chỉ khi `leastInterval(tasks, 1) == len(tasks)`, tức `x ≤ (T + 1) // 2`.

In [36]:
def min_idle(tasks, n):
    return Solution().leastInterval(tasks, n) - len(tasks)

def can_reorganize(s):
    return Solution().leastInterval(list(s), 1) == len(s)

def can_reorganize_oracle(s):
    from itertools import permutations
    return any(all(p[i] != p[i + 1] for i in range(len(p) - 1)) for p in set(permutations(s)))

print(min_idle(list("AAABBB"), 2), min_idle(list("AAAB"), 2), can_reorganize("aab"), can_reorganize("aaab"))
rng = random.Random(152)
for _ in range(600):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 7)))
    assert can_reorganize(s) == can_reorganize_oracle(s), s
    assert can_reorganize(s) == (max(Counter(s).values()) <= (len(s) + 1) // 2)
print("can_reorganize khớp hoán vị vét cạn và điều kiện x <= (T+1)//2")

2 3 True False
can_reorganize khớp hoán vị vét cạn và điều kiện x <= (T+1)//2


### 15.3. Mỗi nhãn một thời gian làm nguội riêng (đọc thêm)

Nếu mỗi nhãn có `cooldown[c]` riêng thì công thức đóng không còn dùng được; ta quay lại **mô phỏng heap + hàng đợi** (Cách 2) với `t_ready = t + cooldown[ch]`, hoặc tham lam theo "việc sẵn sàng có nhiều việc còn lại nhất". (Tính tối ưu của tham lam khi đó **không** đảm bảo nữa — là bài khó hơn.)

### 15.4. Các bài liên quan trên LeetCode (đọc thêm)

- **767. Reorganize String:** xếp để hai ký tự giống nhau không kề nhau (`n = 1`, không idle).
- **358. Rearrange String k Distance Apart (Premium):** hai ký tự giống nhau cách ít nhất `k`; giống bài này nhưng cần dựng chuỗi (heap + hàng đợi).
- **1054. Distant Barcodes:** xếp lại để hai mã vạch kề nhau khác nhau.
- **1405. Longest Happy String:** heap và tham lam theo tần suất.
- **253. Meeting Rooms II / 1834. Single-Threaded CPU:** mô phỏng CPU với heap.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS vét cạn | thử mọi lịch trên trạng thái `(rem, cool)` | bùng nổ | bùng nổ | chỉ làm oracle trên input rất nhỏ |
| 2. Heap + hàng đợi nguội | mỗi khoảng làm việc còn nhiều nhất trong số sẵn sàng | `O(T log k)` | `O(k)` | khi cần **dựng lịch** hoặc cooldown riêng từng nhãn |
| 3. Mô phỏng theo vòng `n + 1` | mỗi vòng làm `n + 1` nhãn khác nhau nhiều nhất | `O(T log k)` | `O(k)` | muốn thấy "khung" rõ ràng |
| 4. Công thức đóng | `max(T, (x-1)(n+1)+s)` | `O(T)` | `O(k)` | **cách nộp bài chuẩn**, ngắn nhất |
| 5. Đếm ô nghỉ | `T + max(0, ô trống - việc lấp)` | `O(T)` | `O(k)` | giải thích bằng "lấp lỗ trống" |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [37]:
print(Solution().leastInterval(list("AABB"), 2))
print(Solution().leastInterval(list("AAAB"), 1))
print(Solution().leastInterval(list("ABCDEF"), 10))
print(Solution().leastInterval(list("AAAAAA"), 0))

5
5
6
6


**Bài 2 (tính tay).** Với `tasks = ["A","A","A","B","B","C"]`, `n = 2`: tìm `x`, `s`, khung xương `(x-1)(n+1)+s`, rồi đáp án. Gán tuple `(x, s, khung_xuong, dap_an)` vào `ans2` (đang là `None`).

In [38]:
ans2 = None      # TODO: tuple 4 số nguyên (x, s, khung_xuong, dap_an)

**Bài 3 (điền TODO).** Hoàn thành `least_interval(tasks, n)` bằng công thức đóng.

In [39]:
def least_interval(tasks, n):
    cnt = Counter(tasks)
    x = None      # TODO: tần suất lớn nhất
    s = None      # TODO: số nhãn có tần suất bằng x
    # TODO: trả về max(số việc, (x - 1) * (n + 1) + s)
    return None

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_least_interval(tasks, n)` theo **Cách 3** (mô phỏng theo vòng `n + 1`, dùng max-heap bằng số âm).

In [40]:
def my_least_interval(tasks, n):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `idle_slots(tasks, n)`: số **ô nghỉ tối thiểu** cần dùng. (b) `is_valid(seq, n)`: cho một lịch `seq` (list nhãn hoặc `"idle"`), kiểm tra **mọi cặp việc cùng nhãn cách nhau ít nhất `n` khoảng** (không cần kiểm tra tối ưu). Ví dụ `is_valid(["A","B","idle","A"], 2)` là `True` (giữa hai `A` có 2 khoảng), `is_valid(["A","B","A"], 2)` là `False`.

In [41]:
def idle_slots(tasks, n):
    # TODO
    pass

def is_valid(seq, n):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: tìm nhãn xuất hiện nhiều nhất và vẽ khung; với `n = 0` không cần đợi.
- Bài 2: đếm: `A` 3 lần, `B` 2 lần, `C` 1 lần.
- Bài 3: đã học ở mục 7.3.
- Bài 4: dùng `heapify([-c for c in Counter(tasks).values()])`.
- Bài 5: số ô nghỉ = đáp án - số việc; `is_valid` cần nhớ lần chạy gần nhất của từng nhãn.

**Mức 2 (vừa):**
- Bài 2: `x = 3`, chỉ `A` đạt 3 nên `s = 1`; khung xương `(3-1)*3 + 1 = 7`; số việc `6`; đáp án `max(6, 7)`.
- Bài 3: `x = max(cnt.values())`, `s = sum(1 for v in cnt.values() if v == x)`.
- Bài 4: mỗi vòng lấy tối đa `n + 1` phần tử khác nhau, giảm 1, cất vào `tmp`, rồi đẩy lại; cộng `n + 1` nếu heap còn, ngược lại cộng số đã lấy.
- Bài 5(b): `last = {}`; với mỗi vị trí `t` có nhãn `x` khác `"idle"`: nếu `x in last` và `t - last[x] - 1 < n` thì `False`; rồi `last[x] = t`.

**Mức 3 (gần đáp án):**
- Bài 3: xem Cách 4.
- Bài 4: xem Cách 3 ở mục 7.2.
- Bài 5(a): `idle_slots = least_interval(tasks, n) - len(tasks)`.

## 19. Đáp án tham khảo

In [42]:
# Bài 1: 5 (A B _ A B), 5 (A B A _ A), 6, 6
assert Solution().leastInterval(list("AABB"), 2) == 5
assert Solution().leastInterval(list("AAAB"), 1) == 5
assert Solution().leastInterval(list("ABCDEF"), 10) == 6
assert Solution().leastInterval(list("AAAAAA"), 0) == 6
print("Bài 1 OK")

# Bài 2
ans2_ref = (3, 1, 7, 7)
x_ref = max(Counter("AAABBC").values())
assert ans2_ref == (x_ref, 1, (x_ref - 1) * 3 + 1, Solution().leastInterval(list("AAABBC"), 2))
assert least_interval_bfs(list("AAABBC"), 2) == 7
print("Bài 2 OK:", ans2_ref)

Bài 1 OK
Bài 2 OK: (3, 1, 7, 7)


In [43]:
# Bài 3
def least_interval_ref(tasks, n):
    cnt = Counter(tasks)
    x = max(cnt.values())
    s = sum(1 for v in cnt.values() if v == x)
    return max(len(tasks), (x - 1) * (n + 1) + s)

rng = random.Random(1903)
for _ in range(1500):
    letters = rng.sample("ABCDEFGH", rng.randint(1, 4))
    tasks = [rng.choice(letters) for _ in range(rng.randint(1, 8))]
    n = rng.randint(0, 5)
    assert least_interval_ref(tasks, n) == least_interval_bfs(tasks, n), (tasks, n)
print("Bài 3 OK")

# Bài 4
def my_least_interval_ref(tasks, n):
    heap = [-c for c in Counter(tasks).values()]
    heapify(heap)
    total = 0
    while heap:
        tmp, taken = [], 0
        for _ in range(n + 1):
            if heap:
                c = heappop(heap) + 1
                taken += 1
                if c:
                    tmp.append(c)
        for c in tmp:
            heappush(heap, c)
        total += n + 1 if heap else taken
    return total

rng = random.Random(1904)
for _ in range(1500):
    letters = rng.sample("ABCDEFGH", rng.randint(1, 4))
    tasks = [rng.choice(letters) for _ in range(rng.randint(1, 8))]
    n = rng.randint(0, 5)
    assert my_least_interval_ref(tasks, n) == least_interval_bfs(tasks, n), (tasks, n)
print("Bài 4 OK")

Bài 3 OK


Bài 4 OK


In [44]:
# Bài 5
def idle_slots_ref(tasks, n):
    return least_interval_ref(tasks, n) - len(tasks)

def is_valid_ref(seq, n):
    last = {}
    for t, x in enumerate(seq):
        if x == "idle":
            continue
        if x in last and t - last[x] - 1 < n:
            return False
        last[x] = t
    return True

assert is_valid_ref(["A", "B", "idle", "A"], 2) is True
assert is_valid_ref(["A", "B", "A"], 2) is False
assert idle_slots_ref(list("AAABBB"), 2) == 2
rng = random.Random(1905)
for _ in range(1500):
    letters = rng.sample("ABCDEFGH", rng.randint(1, 4))
    tasks = [rng.choice(letters) for _ in range(rng.randint(1, 8))]
    n = rng.randint(0, 5)
    assert idle_slots_ref(tasks, n) == least_interval_bfs(tasks, n) - len(tasks)
    seq = schedule_heap(tasks, n)
    assert is_valid_ref(seq, n) == valid_schedule(seq, tasks, n) == True
    # kiểm tra is_valid_ref bắt được lịch sai: ghép hai việc cùng nhãn sát nhau khi n >= 1
    if n >= 1:
        assert is_valid_ref([tasks[0], tasks[0]], n) is False
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template công thức đóng (cách nộp):**

```python
from collections import Counter

class Solution:
    def leastInterval(self, tasks, n):
        cnt = Counter(tasks)
        x = max(cnt.values())                           # tần suất lớn nhất
        s = sum(1 for v in cnt.values() if v == x)      # số nhãn đạt tần suất đó
        return max(len(tasks), (x - 1) * (n + 1) + s)
```

**Template mô phỏng heap (khi cần dựng lịch / cooldown riêng):**

```python
heap = [-c for c in Counter(tasks).values()]; heapify(heap)
cool = deque(); t = 0
while heap or cool:
    t += 1
    if heap:
        c = heappop(heap) + 1
        if c: cool.append((t + n, c))      # chạy lại sớm nhất ở t + n + 1
    if cool and cool[0][0] == t:
        heappush(heap, cool.popleft()[1])
```

**Câu thần chú:** *việc nhiều nhất dựng khung `(x-1)` lần `(n+1)` ô, cộng `s` ô cuối; còn lại lấp ô trống; đáp án không bao giờ nhỏ hơn số việc.*

**Checklist trước khi submit:**

- [ ] Có `max(len(tasks), ...)` để bao trường hợp nhiều việc/không có ô nghỉ?
- [ ] Khung dài `n + 1` (không phải `n`), và dùng `(x - 1)` khung đầy (không phải `x`)?
- [ ] Đã cộng `s` (số nhãn cùng tần suất lớn nhất) chứ không phải `1`?
- [ ] `n = 0` cho ra đúng `len(tasks)`?
- [ ] Đã thử: một việc, toàn nhãn khác nhau, hai nhãn cùng tần suất, một nhãn chiếm áp đảo, `n = 100`?
- [ ] Nếu mô phỏng: vòng/khoảng cuối không cộng ô nghỉ thừa?